# Fabric IQ Service-Principal Auth Diagnostic

This notebook logs in as the **service principal** `AutoFNOL-Logicapps-GraphApp` (the same identity used by the Foundry orchestrator's Fabric IQ tool) and runs three tests against this workspace, in order:

1. **Lakehouse** — query `LH_AutoFNOL` directly via its SQL analytics endpoint.
2. **Fabric Data Agent** — query `DA_AutoFNOL_Ontology` via its published MCP endpoint.
3. **Ontology** — query the `AutoFNOL_Ontology` item directly via its dedicated ontology MCP endpoint.

Purpose: isolate exactly which layer accepts/rejects service-principal (app-only) auth, since delegated/interactive user auth is confirmed to work for all three today.

**Before running:** set the `CLIENT_SECRET` value in the parameters cell below (or wire it to Key Vault via `notebookutils.credentials.getSecret(...)`), then Run All.

In [ ]:
# ---- Parameters: identity + target resource IDs ----
TENANT_ID = "<AAD_TENANT_ID>"
CLIENT_ID = "<AAD_CLIENT_ID>"          # AutoFNOL-Logicapps-GraphApp app (client) id
CLIENT_SECRET = "<AAD_CLIENT_SECRET>"  # fill in manually, or pull from Key Vault below

# Uncomment to pull the secret from Key Vault instead of pasting it above:
# CLIENT_SECRET = notebookutils.credentials.getSecret("https://<your-kv-name>.vault.azure.net/", "AutoFNOL-Logicapps-GraphApp-Secret")

FABRIC_WORKSPACE_ID = "4fb5c773-27f6-4f0d-9eb0-f040abfdd977"   # WS_AutoFNOL
FABRIC_LAKEHOUSE_ID = "8be8ae26-8bf8-493d-8f33-565f4aa7581b"   # LH_AutoFNOL
FABRIC_LAKEHOUSE_SQL_ENDPOINT = "cnfzy3l2lhkuxgxslgdsleid7u-opd3kt7we4gu7hvq6bakx7ozo4.datawarehouse.fabric.microsoft.com"
FABRIC_DATA_AGENT_ID = "f4f24b3a-8cb2-4f33-88ad-04ba57d8be8a"  # DA_AutoFNOL_Ontology
FABRIC_ONTOLOGY_ITEM_ID = "67a24265-e67e-4230-a26c-03075db4147c"  # AutoFNOL_Ontology

TEST_QUESTION = "Show coverage limits for Policy POL-00005"

In [ ]:
# ---- Install/upgrade packages (the pre-installed 'mcp' in Fabric's base image can be an
#      older version lacking streamablehttp_client, so force an upgrade to be safe) ----
%pip install --upgrade --force-reinstall "mcp>=1.9.0" msal pyodbc httpx --quiet

In [ ]:
# ---- Acquire service-principal (app-only, client-credentials) tokens ----
from msal import ConfidentialClientApplication

_app = ConfidentialClientApplication(
    client_id=CLIENT_ID,
    client_credential=CLIENT_SECRET,
    authority=f"https://login.microsoftonline.com/{TENANT_ID}",
)


def get_sp_token(scope: str) -> str:
    result = _app.acquire_token_for_client(scopes=[scope])
    if "access_token" not in result:
        raise RuntimeError(f"Failed to acquire token for {scope}: {result}")
    return result["access_token"]


# Two audiences seen in Fabric docs/samples - acquire both up front so later
# cells can try whichever one is relevant.
fabric_api_token = get_sp_token("https://api.fabric.microsoft.com/.default")
powerbi_api_token = get_sp_token("https://analysis.windows.net/powerbi/api/.default")
sql_token = get_sp_token("https://database.windows.net/.default")

print("Acquired SP tokens for: api.fabric.microsoft.com, analysis.windows.net/powerbi/api, database.windows.net")

## 1. Query the Lakehouse (SQL analytics endpoint)

Directly queries `LH_AutoFNOL` using the service principal's token. This is the control test: it is already confirmed to work outside this notebook, so it should also succeed here.

In [ ]:
import struct
import pyodbc

SQL_COPT_SS_ACCESS_TOKEN = 1256


def sql_connect_as_sp(database: str, token: str):
    token_bytes = token.encode("utf-16-le")
    token_struct = struct.pack("=i", len(token_bytes)) + token_bytes
    conn_str = (
        "DRIVER={ODBC Driver 18 for SQL Server};"
        f"SERVER={FABRIC_LAKEHOUSE_SQL_ENDPOINT},1433;"
        f"DATABASE={database};"
        "Encrypt=yes;"
    )
    return pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct})


try:
    conn = sql_connect_as_sp("LH_AutoFNOL", sql_token)
    cur = conn.cursor()
    cur.execute(
        "SELECT TOP 5 PolicyId, CoverageTypes, LiabilityLimitPerAccident "
        "FROM Policy WHERE PolicyId = 'POL-00005'"
    )
    rows = cur.fetchall()
    print("LAKEHOUSE QUERY RESULT (service principal):")
    for row in rows:
        print("  ", row)
    if not rows:
        print("  (no matching rows - check table/column names for this workspace)")
    lakehouse_test_passed = True
except Exception as e:
    print("LAKEHOUSE QUERY FAILED (service principal):", repr(e))
    lakehouse_test_passed = False

## 2. Query the Fabric Data Agent (MCP endpoint)

Calls the published `DA_AutoFNOL_Ontology` data agent's MCP endpoint with the service principal's token. Per Microsoft's Fabric IQ docs, this is the one MCP endpoint type documented as supporting a direct service-principal (not just delegated) token.

In [ ]:
import asyncio
from mcp import ClientSession

# Defensive import: different mcp package versions have exposed this client
# under different names/paths over time.
try:
    from mcp.client.streamable_http import streamablehttp_client
except ImportError:
    try:
        from mcp.client.streamable_http import streamable_http_client as streamablehttp_client
    except ImportError:
        from mcp.client.streamable_http import streamable_client as streamablehttp_client


def _open_streamable_http(url: str, token: str, timeout: int = 60):
    """The mcp SDK has changed the streamablehttp_client() signature across
    versions (some Fabric runtimes have an older/minimal build that only
    accepts `url`). Rather than guess from the signature, just try each
    known calling convention in turn and use whichever one the installed
    version actually accepts, falling back to a custom httpx client
    factory (which injects the Authorization header at the transport
    level) if none of the direct kwargs are supported."""
    headers = {"Authorization": "Bearer " + token}

    try:
        return streamablehttp_client(url, headers=headers, timeout=timeout)
    except TypeError:
        pass

    try:
        return streamablehttp_client(url, additional_headers=headers, timeout=timeout)
    except TypeError:
        pass

    import httpx

    class _BearerAuth(httpx.Auth):
        def __init__(self, token):
            self.token = token

        def auth_flow(self, request):
            request.headers["Authorization"] = "Bearer " + self.token
            yield request

    try:
        return streamablehttp_client(url, auth=_BearerAuth(token), timeout=timeout)
    except TypeError:
        pass

    # Last resort: inject the header via a custom httpx client factory, which
    # even very old/minimal streamablehttp_client(url) signatures still
    # generally accept as `httpx_client_factory`.
    def _client_factory(*factory_args, **factory_kwargs):
        factory_kwargs.setdefault("headers", {})
        factory_kwargs["headers"] = {**factory_kwargs["headers"], **headers}
        return httpx.AsyncClient(*factory_args, **factory_kwargs)

    try:
        return streamablehttp_client(url, httpx_client_factory=_client_factory)
    except TypeError:
        pass

    # Absolute last resort: call with just the url (no auth injection - will
    # likely fail auth, but at least surfaces a clean error instead of a
    # TypeError about argument count).
    return streamablehttp_client(url)

DATA_AGENT_MCP_URL = (
    f"https://api.fabric.microsoft.com/v1/mcp/workspaces/{FABRIC_WORKSPACE_ID}"
    f"/dataagents/{FABRIC_DATA_AGENT_ID}/agent"
)


async def query_data_agent(question: str, token: str) -> str:
    async with _open_streamable_http(DATA_AGENT_MCP_URL, token) as streams:
        # Different mcp versions yield either (read, write) or
        # (read, write, get_session_id_callback) - handle both.
        read, write = streams[0], streams[1]
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            if not tools.tools:
                return "[no tools returned by data agent]"
            tool = tools.tools[0]
            arg_name = list(tool.inputSchema.get("properties", {}).keys())[0]
            result = await session.call_tool(tool.name, {arg_name: question})
            texts = [getattr(c, "text", str(c)) for c in result.content]
            return f"[isError={result.isError}] " + "\n".join(texts)


try:
    # Fabric/Jupyter kernels already run their own asyncio event loop, so
    # asyncio.run() fails with 'cannot be called from a running event loop'.
    # Use top-level await instead (supported directly by the notebook kernel).
    data_agent_answer = await query_data_agent(TEST_QUESTION, powerbi_api_token)
    print("DATA AGENT RESULT (service principal, powerbi/api token):")
    print(data_agent_answer)
    data_agent_test_passed = "authoriz" not in data_agent_answer.lower() and "not authorized" not in data_agent_answer.lower()
except BaseException as e:
    import traceback

    def _unwrap(exc, depth=0):
        prefix = "  " * depth
        print(f"{prefix}{type(exc).__name__}: {exc}")
        data = getattr(exc, "data", None)
        if data is not None:
            print(f"{prefix}  data: {data!r}")
        for sub in getattr(exc, "exceptions", []) or []:
            _unwrap(sub, depth + 1)

    print("DATA AGENT QUERY FAILED (service principal) - full detail:")
    _unwrap(e)
    traceback.print_exc()
    data_agent_test_passed = False


## 2b. Query the Fabric Data Agent using the notebook's own (user) identity

Same MCP call as above, but the token comes from `notebookutils.credentials.getToken(...)` instead of the service-principal client-credentials flow. When you run this notebook interactively in the Fabric portal, this token represents **your own signed-in user identity** (delegated auth) - not the SP. Compare this cell's result against cell 8 (SP auth) to see whether the Data Agent MCP endpoint accepts user auth but rejects SP auth.

In [ ]:
# ---- Acquire a token as the notebook's own identity (delegated / user auth) ----
# notebookutils.credentials.getToken() returns a token for the identity the notebook
# is actually running as: your signed-in user when run interactively in the portal,
# or the workspace identity / trigger identity when run via a pipeline/schedule.
# notebookutils.credentials is a lazily-registered submodule in some Fabric
# runtime versions, so a plain 'import notebookutils' + attribute access can
# raise AttributeError even though the feature exists. Explicitly import the
# submodule (and fall back to the legacy mssparkutils alias) to be safe.
try:
    from notebookutils import credentials as _nb_credentials
except ImportError:
    from mssparkutils import credentials as _nb_credentials

user_fabric_api_token = _nb_credentials.getToken("https://api.fabric.microsoft.com")
user_powerbi_api_token = _nb_credentials.getToken("pbi")
print("Acquired notebook-identity (user) tokens.")

try:
    user_data_agent_answer = await query_data_agent(TEST_QUESTION, user_fabric_api_token)
    print("DATA AGENT RESULT (notebook/user identity, fabric api token):")
    print(user_data_agent_answer)
    user_data_agent_test_passed = (
        "authoriz" not in user_data_agent_answer.lower()
        and "not authorized" not in user_data_agent_answer.lower()
    )
except BaseException as e:
    import traceback

    def _unwrap(exc, depth=0):
        prefix = "  " * depth
        print(f"{prefix}{type(exc).__name__}: {exc}")
        data = getattr(exc, "data", None)
        if data is not None:
            print(f"{prefix}  data: {data!r}")
        for sub in getattr(exc, "exceptions", []) or []:
            _unwrap(sub, depth + 1)

    print("DATA AGENT QUERY FAILED (notebook/user identity) - full detail:")
    _unwrap(e)
    traceback.print_exc()
    user_data_agent_test_passed = False


## 3. Query the Ontology directly (dedicated ontology MCP endpoint)

Calls the `AutoFNOL_Ontology` item's own MCP endpoint (distinct from the data-agent endpoint above). Per Microsoft's Fabric IQ documentation, this endpoint only supports **delegated** auth (BYO Entra app / managed OAuth) - a service-principal token is expected to fail here. This cell is included to explicitly confirm/document that expected boundary, rather than assume it.

In [ ]:
ONTOLOGY_MCP_URL = (
    f"https://api.fabric.microsoft.com/v1/mcp/dataPlane/workspaces/{FABRIC_WORKSPACE_ID}"
    f"/items/{FABRIC_ONTOLOGY_ITEM_ID}/ontologyEndpoint"
)


async def query_ontology_direct(question: str, token: str) -> str:
    async with _open_streamable_http(ONTOLOGY_MCP_URL, token) as streams:
        # Different mcp versions yield either (read, write) or
        # (read, write, get_session_id_callback) - handle both.
        read, write = streams[0], streams[1]
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            if not tools.tools:
                return "[no tools returned by ontology endpoint]"
            tool = tools.tools[0]
            arg_name = list(tool.inputSchema.get("properties", {}).keys())[0]
            result = await session.call_tool(tool.name, {arg_name: question})
            texts = [getattr(c, "text", str(c)) for c in result.content]
            return f"[isError={result.isError}] " + "\n".join(texts)


try:
    ontology_answer = await query_ontology_direct(TEST_QUESTION, powerbi_api_token)
    print("ONTOLOGY RESULT (service principal):")
    print(ontology_answer)
    ontology_test_passed = True
except BaseException as e:
    import traceback

    def _unwrap(exc, depth=0):
        prefix = "  " * depth
        print(f"{prefix}{type(exc).__name__}: {exc}")
        data = getattr(exc, "data", None)
        if data is not None:
            print(f"{prefix}  data: {data!r}")
        for sub in getattr(exc, "exceptions", []) or []:
            _unwrap(sub, depth + 1)

    print("ONTOLOGY QUERY FAILED (service principal) - full detail (expected per Fabric IQ docs if delegated-only):")
    _unwrap(e)
    traceback.print_exc()
    ontology_test_passed = False


## Summary

In [ ]:
print("===== SERVICE-PRINCIPAL AUTH DIAGNOSTIC SUMMARY =====")
print(f"1. Lakehouse (SQL endpoint) direct query : {'PASS' if lakehouse_test_passed else 'FAIL'}")
print(f"2b. Fabric Data Agent (MCP, notebook/user auth)  : {'PASS' if user_data_agent_test_passed else 'FAIL'}")
print(f"2. Fabric Data Agent (MCP endpoint)      : {'PASS' if data_agent_test_passed else 'FAIL'}")
print(f"3. Ontology (dedicated MCP endpoint)     : {'PASS' if ontology_test_passed else 'FAIL (expected - delegated-only per docs)'}")